[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CityScope/UrbanAccessAnalyzer/blob/main/examples/walkability.ipynb)

# 🚶 Walkability

**How many everyday places can you reach on foot?** This notebook grades every street of a city by its walking distance to the places people need: shops, schools, health care, parks...

### How it works

1. **Streets and places** come from OpenStreetMap: the walking street network, and each type of place (POI) from the Overpass API.
2. **One grade per type of place.** From each POI we walk along the streets. A street scores **100** within the first distance step, and less and less up to the type's `max_distance`; beyond it, **0**.
3. **Points vs polygons.** Shops, schools... are *points*. Parks are *polygons*: people can enter anywhere along their edge, so every street touching the park (or inside it) counts as being at the park.
4. **One final score.** The grades are combined as a mean weighted by each type's `importance` (0 - 100).
5. **Who benefits?** The scores are averaged into H3 hexagons and crossed with population. On the map, each circle is a hexagon: its size is the population and its color the walkability.

| Config field | Meaning |
|---|---|
| `name` | label of the POI type |
| `query` | function that downloads the POIs |
| `geometry` | `"Point"` or `"Polygon"` |
| `importance` | relative weight in the final score |
| `max_distance` | meters walked beyond which the POI is worthless |
| `icon` | emoji shown on the maps |

In [ ]:
# Google Colab only: installs everything (all packages pinned to release v1.0.0). Does nothing locally.
import sys

if "google.colab" in sys.modules:
    !pip install -q "UrbanAccessAnalyzer[plot] @ git+https://github.com/CityScope/UrbanAccessAnalyzer.git@v1.0.0"
    !pip install -q "geohierarchy @ git+https://github.com/CityScope/geohierarchy.git@v1.0.0"
    !pip install -q "pycensus @ git+https://github.com/CityScope/pyCensus.git@v1.0.0"
    # If an import fails afterwards: Runtime > Restart session, then run again.

In [ ]:
import os
import sys

from UrbanAccessAnalyzer import grading, osm_io, workflows

# 1 Inputs

In [ ]:
# USER INPUT
place = "Alcoy, España"
download_buffer = 1500  # meters of extra area for streets and POIs (at least the largest max_distance)
n_steps = 5  # distance steps per POI type: with 5, scores are 100, 80, 60, 40, 20, then 0
h3_resolution = 10  # hexagon size: 10 is ~0.015 km2, 9 is ~0.1 km2
population = "worldpop"  # "worldpop": 100 m grid, anywhere in the world | "census": the country's own census (an error if not implemented)

## 1.1 Points of interest

In [ ]:
# USER INPUT. One entry per type of place. "icon" is the emoji shown on the maps (any emoji).
# Ready-made queries in osm_io: schools, healthcare, groceries, shops, restaurants, libraries,
# pharmacies, gyms, cinemas, bus_stops, green_areas. Your own data or OSM tags: see 1.2.
pois = [
    {"name": "schools",     "query": osm_io.schools,     "geometry": "Point",   "importance": 10, "max_distance": 1000, "icon": "🏫"},
    {"name": "groceries",   "query": osm_io.groceries,   "geometry": "Point",   "importance": 8,  "max_distance": 1500, "icon": "🛒"},
    {"name": "restaurants", "query": osm_io.restaurants, "geometry": "Point",   "importance": 7,  "max_distance": 1500, "icon": "🍽️"},
    {"name": "healthcare",  "query": osm_io.healthcare,  "geometry": "Point",   "importance": 3,  "max_distance": 2000, "icon": "🏥"},
    {"name": "parks",       "query": osm_io.green_areas, "geometry": "Polygon", "importance": 5,  "max_distance": 1500, "icon": "🌳"},
    {"name": "bakeries",    "query": osm_io.tag_query("shop", "bakery"), "geometry": "Point", "importance": 4, "max_distance": 1000, "icon": "🥖"},
]

## 1.2 Your own points of interest (optional)

Another type of place is one more entry in `pois`. Two ways to get its data.

### A. A file of your own

Points or polygons, in any format GeoPandas reads (`.gpkg`, `.geojson`, `.shp`, `.parquet`...) and any coordinate system (it must have one). An optional column holds the names shown on the maps.

- **Locally:** put the file next to the notebook and set `my_file` to its path.
- **Google Colab:** leave `my_file = ""`. When you run the cell, a **window opens to choose the file from your computer** (Colab keeps it until the session ends).

In [ ]:
# USER INPUT
use_my_file = False  # True: add your own file of points or polygons
my_file = ""  # local path. In Google Colab leave it empty: a file picker opens automatically

if use_my_file:
    pois.append({
        "name": "my_pois",
        "query": grading.query_from_file(my_file, name_column=None),  # name_column: column with the names, if any
        "geometry": "Point",  # "Polygon" if your file has polygons (parks, plazas...)
        "importance": 5,
        "max_distance": 1000,
        "icon": "⭐",
    })

### B. A custom OpenStreetMap query

OpenStreetMap describes every place with tags (`key=value`). To find the tags of what you want, browse the [map features](https://wiki.openstreetmap.org/wiki/Map_features), search a value on [taginfo](https://taginfo.openstreetmap.org/), or click any place on [openstreetmap.org](https://www.openstreetmap.org) with the *query features* tool.

- **One tag:** `osm_io.tag_query("amenity", "dentist")` (use `"*"` as the value for any value).
- **Several tags or conditions:** write the [Overpass](https://overpass-turbo.eu/) query yourself, as in the function below. `{{bbox}}` is replaced by the download area.

#### Let an AI write the query for you

Copy this prompt into an AI assistant, changing `"hospital"` for what you need:

> Modify the following Python function so that it searches for `"hospital"` instead of dentists using an OpenStreetMap Overpass API query.
>
> Ensure the function:
>
> - Keeps the same structure and return behavior
> - Uses the bounding box placeholder `{{bbox}}`
> - Returns `osm_io.overpass_query(query, bounds)`
>
> Original function:
>
> ```python
> def dentists(bounds):
>     query = """
>         [out:json][timeout:25];
>         nwr["amenity"="dentist"]({{bbox}});
>         (._;>;);
>         out body;
>     """
>     return osm_io.overpass_query(query, bounds)
> ```

In [ ]:
def dentists(bounds):
    query = """
        [out:json][timeout:25];
        nwr["amenity"="dentist"]({{bbox}});
        (._;>;);
        out body;
    """
    return osm_io.overpass_query(query, bounds)


# Uncomment to add it ("Polygon" for areas such as parks)
# pois.append({"name": "dentists", "query": dentists, "geometry": "Point", "importance": 3, "max_distance": 1000, "icon": "🦷"})

# 2 Run

In [ ]:
result = workflows.walkability(place, pois, download_buffer=download_buffer, n_steps=n_steps,
                               h3_resolution=h3_resolution, population=population)
result.map  # the walkability map, also saved in output/<place>/maps/ as a standalone .html: double-click it

In [ ]:
result.poi_map  # the places found (map in the same folder)

In [ ]:
result.stats  # people per walkability band

In [ ]:
if "google.colab" in sys.modules:
    !zip -r /content/output.zip output  # then download it from the file browser

Saved in `output/<place>/` (`result.folder`):

- `maps/points_of_interest.html` and `maps/walkability_map.html`: interactive maps, double-click to open them in your browser
- `access_streets.gpkg`: streets with one score per POI type and the final `access_score`
- `stats.csv`: people per walkability band